        # 0002 — BRH document inventory: discovery & volume

        **Question.** How do we enumerate every document on `brh.ht`, how big is
        the corpus, and how do we ask a schedule "what changed since last time?"

        **Options (discovery).**
        1. WordPress REST API (`/wp-json/wp/v2/media`) — structured, paginated
        2. HTML crawl of `/publications/`
        3. Firecrawl `/map`
        4. `sitemap.xml` (reported empty)

        **Metric.** Documents found, whether dates/URLs come for free, requests
        needed, estimated bytes on disk, and whether an incremental filter exists.

        **Feeds.** `docs/plans/2026-10-14-brh-inventory-design.md`, the future
        discovery ADR, `brh_core/ingestion/discover.py`.

        This notebook makes **live, polite requests** (identified UA, low volume).
        


In [ ]:
        import time

        import httpx
        import pandas as pd

        UA = "brh-intelligence-research/0.1 (+non-training public-sector RAG inventory)"
        HOST = "https://www.brh.ht"
        API = f"{HOST}/wp-json/wp/v2"

        client = httpx.Client(
            headers={"User-Agent": UA}, timeout=30.0, follow_redirects=True
        )

        r = client.get(f"{API}/media", params={"per_page": 1})
        print("API reachable:", r.status_code, "| namespaces ok:", bool(r.headers.get("x-wp-total")))
        


## 1. Crawl permission

`robots.txt` decides whether we may do this at all.


In [ ]:
        robots = client.get(f"{HOST}/robots.txt")
        print("status:", robots.status_code)
        print(robots.text[:500])
        


## 2. Discovery surface (D1)

Which source enumerates documents, and what does it cost us?


In [ ]:
        def total(path: str, **params) -> dict:
            resp = client.get(f"{API}/{path}", params={"per_page": 1, **params})
            count = resp.headers.get("x-wp-total")
            return {
                "endpoint": f"{path}?{'&'.join(f'{k}={v}' for k, v in params.items())}",
                "status": resp.status_code,
                "count": int(count) if count else None,
            }


        coverage = pd.DataFrame(
            [
                total("media", mime_type="application/pdf"),
                total("media", media_type="application"),
                total("posts"),
                total("pages"),
            ]
        )
        coverage
        


        ## 3. Incremental (D4)

        A scheduled job must ask *"what changed since the last run?"* — not re-list
        everything. WordPress supports `modified_after` / `after` on post-type
        endpoints.
        


In [ ]:
        since = "2026-10-01T00:00:00"
        changed = client.get(
            f"{API}/media",
            params={"mime_type": "application/pdf", "modified_after": since, "per_page": 1},
        )
        print(f"PDFs modified since {since}: {changed.headers.get('x-wp-total')} of 1171")

        newest = client.get(
            f"{API}/media",
            params={
                "mime_type": "application/pdf",
                "orderby": "modified",
                "order": "desc",
                "per_page": 5,
                "_fields": "id,modified_gmt,slug,source_url",
            },
        ).json()
        pd.DataFrame(
            [
                {
                    "wp_id": m["id"],
                    "modified_gmt": m["modified_gmt"],
                    "slug": m["slug"],
                    "source_url": m["source_url"],
                }
                for m in newest
            ]
        )
        


        ## 4. Volume + change-detection surface (D5, D2, D6)

        Sample the first page of PDFs, `HEAD` each one, and read what the server
        tells us. This gives an order-of-magnitude storage estimate and reveals
        whether cheap change detection (ETag / Last-Modified) is available.
        


In [ ]:
        SAMPLE = 25
        sample = client.get(
            f"{API}/media",
            params={
                "mime_type": "application/pdf",
                "per_page": SAMPLE,
                "_fields": "id,source_url,slug",
            },
        ).json()

        rows = []
        for item in sample:
            t0 = time.perf_counter()
            try:
                head = client.head(item["source_url"])
                length = int(head.headers.get("content-length", 0))
                row = {
                    "slug": item["slug"][:32],
                    "status": head.status_code,
                    "mb": round(length / 1e6, 2),
                    "seconds": round(time.perf_counter() - t0, 2),
                    "last_modified": head.headers.get("last-modified"),
                    "etag": head.headers.get("etag"),
                    "accept_ranges": head.headers.get("accept-ranges"),
                    "source_url": item["source_url"],
                }
            except Exception as exc:  # noqa: BLE001
                row = {"slug": item["slug"][:32], "status": -1, "mb": 0.0,
                       "seconds": round(time.perf_counter() - t0, 2),
                       "last_modified": None, "etag": None,
                       "accept_ranges": None, "source_url": item["source_url"],
                       "error": str(exc)[:60]}
            rows.append(row)
            time.sleep(0.4)  # be polite

        df = pd.DataFrame(rows)
        ok = df[df["status"] == 200]
        mean_mb = ok["mb"].mean()
        print(f"sampled {len(df)}  ok={len(ok)}")
        print(f"size: mean={mean_mb:.2f} MB  median={ok['mb'].median():.2f} MB  max={ok['mb'].max():.2f} MB")
        print(f"estimated corpus ≈ {mean_mb * 1171 / 1000:.1f} GB   (mean × 1171 PDFs)")
        print(f"ETag present on any?      {ok['etag'].notna().any()}")
        print(f"Last-Modified on all?     {ok['last_modified'].notna().all()}")
        df
        


## 5. Fetch + identity (D2, D3)

Actually download the smallest PDF and hash it — the content-addressing key.


In [ ]:
        import hashlib


        def download_and_hash(url: str) -> tuple[str, int]:
            digest = hashlib.sha256()
            size = 0
            with client.stream("GET", url) as stream:
                stream.raise_for_status()
                for chunk in stream.iter_bytes():
                    digest.update(chunk)
                    size += len(chunk)
            return digest.hexdigest(), size


        smallest = ok.nsmallest(1, "mb").iloc[0]
        print(f"downloading {smallest['slug']} ({smallest['mb']} MB) …")
        sha, nbytes = download_and_hash(smallest["source_url"])
        print(f"sha256 = {sha}")
        print(f"bytes  = {nbytes}  ({nbytes / 1e6:.2f} MB, matches HEAD: {abs(nbytes - smallest['mb'] * 1e6) < 5000})")
        


        ## Decision

        - **D1 Discovery = WordPress REST API.** `wp/v2/media?mime_type=application/pdf`
          enumerates the whole corpus (1171 PDFs) as JSON with `source_url`,
          `date`, `modified_gmt`, and `id` — no HTML parsing, no crawl, no
          paid crawler. `media_type=application/pdf` is invalid (the sibling doc
          was wrong); the correct filter is `mime_type=application/pdf`.
        - **D4 Incremental = `modified_after`.** One query with the watermark from
          the last run returns only changed documents. This is the scheduling unit.
        - **D2 Identity = `sha256(bytes)`.** The server gives `Last-Modified` and
          `Content-Length` but **no `ETag`**, so pre-download checks are
          `If-Modified-Since` + size (a hint, not proof). The definitive identity
          is the hash, computed after download. `wp_id` / `source_url` are
          provenance, not identity — the same bytes can appear at two URLs.
        - **D3 Fetching = `httpx` streaming.** Files are plain `application/pdf`
          over Apache; no JS, no browser. Firecrawl (sibling ADR-0031) is not
          needed for the document inventory.
        - **D5 Volume.** See the estimate above — if it is multi-GB, backfill in
          batches and confirm the storage target before the full pull.
        - **D6 Politeness.** ~0.4 s delay, one HEAD/GET at a time, identified UA.
          Measure a safe concurrency in experiment 0003/0004 before bulk download.

        **Next experiments:** 0003 identity & re-download correctness
        (same bytes at two URLs, `If-Modified-Since` behaviour), 0004 bulk
        throughput + politeness at scale. Then the ADRs, then code.
        
